# ACL Tear Detection - Training Notebook
**Dataset:** Priyank Saxena's ACL MRI Dataset (Sagittal Views)

This notebook trains a CNN to classify ACL tears using transfer learning and 2D slice-based approach.

## Setup
1. Upload `processed_sagittal_resized` folder to Google Drive
2. Run this notebook in Colab with GPU runtime

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuration - UPDATE THIS PATH
DATA_DIR = '/content/drive/MyDrive/processed_sagittal_resized'  # Change to your path

# Training settings
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
NUM_CLASSES = 3  # 0: Normal, 1: Partial, 2: Complete (set to 2 for binary)
BINARY_MODE = False  # Set True to combine Partial+Complete as "Tear"
CENTER_SLICES = 20  # Use center N slices (ACL region)
RANDOM_SEED = 42

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Load and Explore Data

In [ ]:
# Load metadata
data_path = Path(DATA_DIR)
metadata = pd.read_csv(data_path / 'metadata.csv')

print(f"Total patients: {len(metadata)}")
print(f"\nLabel distribution:")
print(metadata['label_name'].value_counts())

# Convert labels for binary mode if needed
if BINARY_MODE:
    metadata['label'] = (metadata['label'] > 0).astype(int)  # 0: Normal, 1: Tear
    NUM_CLASSES = 2
    print(f"\nBinary mode - New distribution:")
    print(metadata['label'].value_counts())

In [ ]:
# Split data: patient-level split to avoid data leakage
train_df, temp_df = train_test_split(
    metadata, test_size=0.3, stratify=metadata['label'], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=RANDOM_SEED
)

print(f"Train: {len(train_df)} patients")
print(f"Val: {len(val_df)} patients")
print(f"Test: {len(test_df)} patients")

print(f"\nTrain label distribution:")
print(train_df['label'].value_counts())

## 2. Dataset and Data Augmentation

In [ ]:
class ACLSliceDataset(Dataset):
    """
    Dataset that treats each slice as a separate sample.
    Uses center slices where ACL is typically visible.
    """
    def __init__(self, df, data_dir, transform=None, center_slices=20):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.center_slices = center_slices
        
        # Pre-compute slice indices
        self.samples = []  # (patient_idx, slice_idx, label)
        for idx, row in self.df.iterrows():
            num_slices = int(row['num_slices'])
            # Get center slices
            start = max(0, (num_slices - center_slices) // 2)
            end = min(num_slices, start + center_slices)
            for slice_idx in range(start, end):
                self.samples.append((idx, slice_idx, int(row['label'])))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        patient_idx, slice_idx, label = self.samples[idx]
        row = self.df.iloc[patient_idx]
        
        # Load volume
        file_path = self.data_dir / row['filename']
        volume = np.load(file_path)['data']
        
        # Get slice and normalize to 0-1
        img = volume[slice_idx].astype(np.float32) / 255.0
        
        # Convert to 3-channel for pretrained models
        img = np.stack([img, img, img], axis=0)  # (3, H, W)
        img = torch.from_numpy(img)
        
        if self.transform:
            img = self.transform(img)
        
        return img, label, patient_idx
    
    def get_labels(self):
        """Return labels for all samples (for weighted sampling)."""
        return [s[2] for s in self.samples]

In [ ]:
# Data augmentation for training
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet stats
])

val_transform = transforms.Compose([
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = ACLSliceDataset(train_df, DATA_DIR, train_transform, CENTER_SLICES)
val_dataset = ACLSliceDataset(val_df, DATA_DIR, val_transform, CENTER_SLICES)
test_dataset = ACLSliceDataset(test_df, DATA_DIR, val_transform, CENTER_SLICES)

print(f"Train samples (slices): {len(train_dataset)}")
print(f"Val samples (slices): {len(val_dataset)}")
print(f"Test samples (slices): {len(test_dataset)}")

In [ ]:
# Weighted sampler to handle class imbalance
train_labels = train_dataset.get_labels()
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print(f"Class counts: {class_counts}")
print(f"Class weights: {class_weights}")

In [ ]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 3. Model Architecture

In [ ]:
class ACLClassifier(nn.Module):
    """
    Transfer learning model using pretrained EfficientNet-B0.
    """
    def __init__(self, num_classes=3, dropout=0.5):
        super(ACLClassifier, self).__init__()
        
        # Load pretrained EfficientNet-B0
        self.backbone = models.efficientnet_b0(weights='IMAGENET1K_V1')
        
        # Freeze early layers (optional - can unfreeze for fine-tuning)
        for param in list(self.backbone.parameters())[:-20]:
            param.requires_grad = False
        
        # Replace classifier head
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout/2),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

# Create model
model = ACLClassifier(num_classes=NUM_CLASSES).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 4. Training Setup

In [ ]:
# Loss function with class weights
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels, _ in tqdm(loader, desc='Training', leave=False):
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / total, 100. * correct / total


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(loader, desc='Validating', leave=False):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    return running_loss / total, 100. * correct / total, all_preds, all_labels, all_probs

## 5. Training Loop

In [ ]:
# Training history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

best_val_acc = 0
patience_counter = 0
early_stop_patience = 10

print("Starting training...\n")

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc, _, _, _ = validate(model, val_loader, criterion, device)
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_acl_model.pth')
        print(f"  ✓ New best model saved! (Val Acc: {val_acc:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break
    
    print()

## 6. Training Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Acc', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## 7. Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_acl_model.pth'))

# Evaluate on test set (slice-level)
test_loss, test_acc, test_preds, test_labels, test_probs = validate(
    model, test_loader, criterion, device
)

print(f"Test Accuracy (slice-level): {test_acc:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
# Classification report
if BINARY_MODE:
    target_names = ['Normal', 'Tear']
else:
    target_names = ['Normal', 'Partial', 'Complete']

print("\nClassification Report (Slice-Level):")
print(classification_report(test_labels, test_preds, target_names=target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Slice-Level)')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 8. Patient-Level Prediction (Majority Voting)

In [ ]:
def patient_level_prediction(model, dataset, device):
    """
    Aggregate slice predictions to patient-level using majority voting and probability averaging.
    """
    model.eval()
    
    patient_preds = {}  # patient_idx -> list of predictions
    patient_probs = {}  # patient_idx -> list of probabilities
    patient_labels = {}  # patient_idx -> true label
    
    loader = DataLoader(dataset, batch_size=32, shuffle=False)
    
    with torch.no_grad():
        for images, labels, patient_idxs in tqdm(loader, desc='Patient prediction'):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = outputs.max(1)
            
            for i, patient_idx in enumerate(patient_idxs.numpy()):
                if patient_idx not in patient_preds:
                    patient_preds[patient_idx] = []
                    patient_probs[patient_idx] = []
                    patient_labels[patient_idx] = labels[i].item()
                
                patient_preds[patient_idx].append(preds[i].item())
                patient_probs[patient_idx].append(probs[i].cpu().numpy())
    
    # Aggregate predictions
    final_preds = []
    final_labels = []
    final_probs = []
    
    for patient_idx in sorted(patient_preds.keys()):
        # Majority voting
        preds = patient_preds[patient_idx]
        majority_pred = max(set(preds), key=preds.count)
        
        # Average probabilities
        avg_prob = np.mean(patient_probs[patient_idx], axis=0)
        
        final_preds.append(majority_pred)
        final_labels.append(patient_labels[patient_idx])
        final_probs.append(avg_prob)
    
    return final_preds, final_labels, final_probs

# Get patient-level predictions
patient_preds, patient_labels, patient_probs = patient_level_prediction(
    model, test_dataset, device
)

patient_acc = 100 * np.mean(np.array(patient_preds) == np.array(patient_labels))
print(f"\nPatient-Level Accuracy: {patient_acc:.2f}%")

In [ ]:
# Patient-level classification report
print("\nClassification Report (Patient-Level):")
print(classification_report(patient_labels, patient_preds, target_names=target_names))

# Patient-level confusion matrix
cm_patient = confusion_matrix(patient_labels, patient_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_patient, annot=True, fmt='d', cmap='Greens',
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Patient-Level)')
plt.tight_layout()
plt.savefig('confusion_matrix_patient.png', dpi=150)
plt.show()

## 9. Save Final Model

In [ ]:
# Save model to Drive
import shutil

save_path = Path(DATA_DIR).parent / 'acl_model_final.pth'
shutil.copy('best_acl_model.pth', save_path)
print(f"Model saved to: {save_path}")

# Save training history
history_df = pd.DataFrame(history)
history_df.to_csv(Path(DATA_DIR).parent / 'training_history.csv', index=False)
print("Training history saved.")

## 10. Quick Inference Example

In [ ]:
def predict_single_volume(model, volume_path, device, transform=None):
    """
    Predict ACL status for a single MRI volume.
    """
    model.eval()
    
    # Load volume
    volume = np.load(volume_path)['data']
    num_slices = volume.shape[0]
    
    # Get center slices
    start = max(0, (num_slices - CENTER_SLICES) // 2)
    end = min(num_slices, start + CENTER_SLICES)
    
    all_probs = []
    
    with torch.no_grad():
        for i in range(start, end):
            img = volume[i].astype(np.float32) / 255.0
            img = np.stack([img, img, img], axis=0)
            img = torch.from_numpy(img).unsqueeze(0)
            
            if transform:
                img = transform(img.squeeze(0)).unsqueeze(0)
            
            img = img.to(device)
            output = model(img)
            prob = torch.softmax(output, dim=1)
            all_probs.append(prob.cpu().numpy()[0])
    
    # Average probabilities
    avg_prob = np.mean(all_probs, axis=0)
    prediction = np.argmax(avg_prob)
    
    return prediction, avg_prob

# Example usage
sample_file = list(Path(DATA_DIR).glob('*.npz'))[0]
pred, probs = predict_single_volume(model, sample_file, device, val_transform)

print(f"\nSample prediction for: {sample_file.name}")
print(f"Predicted class: {target_names[pred]}")
print(f"Probabilities: {dict(zip(target_names, probs.round(3)))}")

---
## Summary

This notebook implements:
- 2D slice-based classification with center slice selection
- Transfer learning with EfficientNet-B0
- Data augmentation (flip, rotation, affine, color jitter)
- Class imbalance handling (weighted sampling + weighted loss)
- Patient-level aggregation via majority voting
- Early stopping and learning rate scheduling

**Next steps to improve:**
1. Try 3D CNN (e.g., 3D ResNet) if more GPU memory available
2. Use attention mechanisms to focus on ACL region
3. Ensemble multiple models
4. Collect more Normal class samples